In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv("data/processed/final_feature_matrix.csv", index_col=0, parse_dates=True)
stationary_vars = df[["d_log_eua", "ttf_gas", "coal_price", "renewable_share"]].diff().dropna()

split = int(len(stationary_vars) * 0.80)
train_var, test_var = stationary_vars.iloc[:split], stationary_vars.iloc[split:]

# 1. VAR Model
var_model = VAR(train_var)
var_results = var_model.fit(maxlags=5, ic="aic")
print(var_results.summary())

# 2. ARIMAX Model
y_train = stationary_vars["d_log_eua"].iloc[:split]
X_train = stationary_vars[["ttf_gas", "coal_price", "renewable_share"]].iloc[:split]
X_test  = stationary_vars[["ttf_gas", "coal_price", "renewable_share"]].iloc[split:]

arimax = SARIMAX(y_train, exog=X_train, order=(2, 0, 1)).fit(disp=False)
print(arimax.summary())

# Save predictions for final evaluation
arimax_preds = arimax.forecast(steps=len(test_var), exog=X_test)
arimax_preds.to_csv("data/processed/baseline_arimax_preds.csv")